In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from matplotlib.collections import LineCollection

In [ ]:
# Brillouin Zone information
MAX_KX_BZ = 4.0 * np.pi / (3 * np.sqrt(3.0))
MAX_KY_BZ = 2.0 * np.pi / 3.0


def plotFirstBrillouinZoneBoundary(ax = None):
    brillouinZoneVertices = np.zeros((7, 2))  # One more to close the polygon

    brillouinZoneVertices[:, 0] = np.array(
        [
            4.0 * np.pi / (3 * np.sqrt(3.0)),
            2.0 * np.pi / (3 * np.sqrt(3.0)),
            -2.0 * np.pi / (3 * np.sqrt(3.0)),
            -4.0 * np.pi / (3 * np.sqrt(3.0)),
            -2.0 * np.pi / (3 * np.sqrt(3.0)),
            2.0 * np.pi / (3 * np.sqrt(3.0)),
            4.0 * np.pi / (3 * np.sqrt(3.0)),
        ]
    )

    brillouinZoneVertices[:, 1] = np.array(
        [
            0.0,
            -2.0 * np.pi / 3.0,
            -2.0 * np.pi / 3.0,
            0.0,
            2.0 * np.pi / 3.0,
            2.0 * np.pi / 3.0,
            0.0,
        ]
    )
    if ax == None:
        ax = plt.gca()
    ax.plot(
        brillouinZoneVertices[:, 0],
        brillouinZoneVertices[:, 1],
        "--",
        color="black",
        linewidth=2,
    )

In [ ]:
# Plotting convenience
plt.rcParams["text.usetex"] = True
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Computer Modern Roman"
plt.rcParams["font.sans-serif"] = "Computer Modern Sans serif"
plt.rcParams["font.monospace"] = "Computer Modern Typewriter"
plt.rcParams["axes.titlesize"] = 36
plt.rcParams["axes.labelsize"] = 36
plt.rcParams["xtick.labelsize"] = 32
plt.rcParams["ytick.labelsize"] = 32
plt.rcParams["font.size"] = 32
plt.rcParams["legend.fontsize"] = 32
plt.rcParams["legend.title_fontsize"] = 32
# Optionally, add custom LaTeX preamble
plt.rcParams["text.latex.preamble"] = (
    r"\usepackage{amsmath} \usepackage{amsfonts} \usepackage{amssymb}"
)

# Set rcParams for tighter layout
plt.rcParams["figure.autolayout"] = True
plt.rcParams["figure.constrained_layout.use"] = False
plt.rcParams["axes.linewidth"] = 1.2

# Set rcParams to show ticks on both left and right sides
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.bottom"] = True
plt.rcParams["ytick.left"] = True
plt.rcParams["xtick.top"] = True
plt.rcParams["ytick.right"] = True

plt.rcParams["axes.xmargin"] = 0.01

In [ ]:
# Defining sympy symbols
k_x, k_y = sp.symbols('k_x k_y', real=True)
a  = sp.symbols('a', real=True, positive=True)
t_pi, t_sigma = sp.symbols('t_pi t_sigma', real=True) #t_I and t_D respectively in Fortran programs
l_soc = sp.symbols('lambda_soc', real=True)
d_tri = sp.symbols('delta_tri', real=True)
t_rashba = sp.symbols('t_rashba', real=True)
mu = sp.symbols('mu', real=True)

In [ ]:
# Pauli matrices
s_0 = sp.eye(2)
s_x = sp.Matrix([[0, 1], [1, 0]])
s_y = sp.Matrix([[0, -sp.I], [sp.I, 0]])
s_z = sp.Matrix([[1, 0], [0, -1]])

# Orbital angular momentum matrices in [111] direction (z || [111])
L_x = sp.Matrix([
    [0, 0, sp.I],
    [0, 0, sp.I],
    [-sp.I, -sp.I, 0]
]) / sp.sqrt(2)

L_y = sp.Matrix([
    [0, -2*sp.I, -sp.I],
    [2*sp.I, 0, sp.I],
    [sp.I, -sp.I, 0]
]) / sp.sqrt(6)

L_z = sp.Matrix([
    [0, -sp.I, sp.I],
    [sp.I, 0, -sp.I],
    [-sp.I, sp.I, 0]
]) / sp.sqrt(3)


# Building Hamiltonian
## Kinetic term

In [ ]:
# Vectors connecting nearest neighbors - TODO: add plot of structure
k1 = k_y
k2 = sp.sqrt(3) / 2 * k_x - sp.Rational(1, 2) * k_y
k3 = -sp.sqrt(3) / 2 * k_x - sp.Rational(1, 2) * k_y


eps_yz = -t_sigma * (sp.exp(sp.I * k1) + sp.exp(sp.I * k2)) - t_pi * sp.exp(sp.I * k3)
eps_zx = -t_sigma * (sp.exp(sp.I * k1) + sp.exp(sp.I * k3)) - t_pi * sp.exp(sp.I * k2)
eps_xy = -2 * t_sigma * sp.cos(sp.sqrt(3) / 2 * k_x) * sp.exp(-sp.I / 2 * k_y) - t_pi * sp.exp(sp.I * k_y)

H_intraorbital = sp.Matrix([[eps_yz, 0, 0], [0, eps_zx, 0], [0, 0, eps_xy]])
# Extend to sublatttice basis
H_intraorbital = sp.Matrix(sp.kronecker_product(s_x, H_intraorbital))

# Extend to spin basis
H_intraorbital = sp.Matrix(sp.kronecker_product(s_0, H_intraorbital))

display(H_intraorbital)

## Interorbital hoppings

In [ ]:
# Compute odd-momentum, interorbital hoppping
xi_yz_zx = -t_rashba * (2 * sp.I * sp.exp(-sp.I * k1 / 2) * sp.sin(sp.sqrt(3) / 2 * k_x))
xi_yz_xy = -t_rashba * (sp.exp(sp.I * k1) - sp.exp(sp.I * k3))
xi_zx_xy = -t_rashba * (sp.exp(sp.I * k1) - sp.exp(sp.I * k2))

H_interorbital = sp.Matrix([[0, xi_yz_zx, xi_yz_xy], [-xi_yz_zx, 0, xi_zx_xy], [-xi_yz_xy, -xi_zx_xy, 0]])
# Extend to sublatttice basis
H_interorbital = sp.Matrix(sp.kronecker_product(s_x, H_interorbital))

# Extend to spin basis
H_interorbital = sp.Matrix(sp.kronecker_product(s_0, H_interorbital))
display(H_interorbital)

## Trigonal crystal field at the interface

In [ ]:
# Compute trigonal field
H_tri = d_tri / 2 * sp.Matrix([
    [0, 1, 1],
    [1, 0, 1],
    [1, 1, 0]
])
H_tri = sp.kronecker_product(s_0, sp.kronecker_product(s_0, H_tri))
display(H_tri)

## Atomic spin-orbit coupling $\vec{L} \cdot \vec{S}$

In [ ]:
# Compute atomic L \cdot S coupling
H_SOC = l_soc * (sp.Matrix(sp.kronecker_product(s_z, sp.kronecker_product(s_0, L_z))) +
               sp.Matrix(sp.kronecker_product(s_y, sp.kronecker_product(s_0, L_y))) +
               sp.Matrix(sp.kronecker_product(s_x, sp.kronecker_product(s_0, L_x))))
display(H_SOC)

## Chemical potential

In [ ]:
H_mu = -mu * sp.eye(12)
display(H_mu)

## Export a callback function that will create our Hamiltonian automatically, based on parameters. This will enable conducting numerical calculations with ease.

In [ ]:
#No need to conjugate since H is diagonalized via eigh(), which take only upper triangle
H = H_intraorbital + H_interorbital + H_SOC + H_tri + H_mu
compute_H = sp.lambdify((k_x, k_y, t_sigma, t_pi, l_soc, t_rashba, d_tri, mu), H, modules='numpy')

In [ ]:
#defining parameters
from dataclasses import dataclass
@dataclass
class dispersion_parameters:
    t_sigma: np.float64 = np.float64(500) #[meV]
    t_pi: np.float64 = np.float64(40) #[meV]
    l_soc: np.float64 = np.float64(-10) #[meV]
    t_rashba: np.float64 = np.float64(5) #[meV]
    d_tri: np.float64 = np.float64(10) #[meV]
    mu: np.float64 = np.float64(0.) #[meV]
    a: np.float64 = np.float64(1.0)

params = dispersion_parameters() # default values

In [ ]:
def calculate_spin_expectation(eigenvectors: np.ndarray) -> tuple:
    """
    Calculate spin expectation values <S_x>, <S_y>, <S_z> for each eigenvector.

    Parameters:
    -----------
    eigenvectors : np.ndarray
        Array of shape (N, N) where each column is an eigenvector.
        First N/2 indices correspond to spin-up, second N/2 to spin-down.

    Returns:
    --------
    spin_x, spin_y, spin_z : np.ndarray
        Arrays of shape (N,) containing expectation values for each eigenvector.
    """
    N = eigenvectors.shape[0]
    n_orb = N // 2
    n_states = eigenvectors.shape[1]

    spin_x = np.zeros(n_states)
    spin_y = np.zeros(n_states)
    spin_z = np.zeros(n_states)

    for i in range(n_states):
        psi = eigenvectors[:, i]
        psi_up = psi[:n_orb]
        psi_down = psi[n_orb:]

        # <S_x> = (ℏ/2) * (psi_up† psi_down + psi_down† psi_up)
        spin_x[i] = 0.5 * (np.vdot(psi_up, psi_down) + np.vdot(psi_down, psi_up)).real

        # <S_y> = (ℏ/2) * i * (psi_up† psi_down - psi_down† psi_up)
        spin_y[i] = 0.5 * (np.vdot(psi_up, psi_down) - np.vdot(psi_down, psi_up)).imag

        # <S_z> = (ℏ/2) * (|psi_up|² - |psi_down|²)
        spin_z[i] = 0.5 * (np.vdot(psi_up, psi_up) - np.vdot(psi_down, psi_down)).real

    return spin_x, spin_y, spin_z

In [ ]:
def plot_k_slices(params: dispersion_parameters, ham_dim: int, n_k_points: int = 200):
    fig, axes = plt.subplots(2,2, figsize=(20, 20))
    k_x_vals = np.linspace(-MAX_KX_BZ, MAX_KX_BZ, n_k_points)
    Energies = np.zeros((k_x_vals.size, ham_dim))
    S_z = np.zeros((k_x_vals.size, ham_dim))

    for i, kx in enumerate(k_x_vals):
        H_k = compute_H(kx, 0, params.t_sigma, params.t_pi, params.l_soc, params.t_rashba, params.d_tri, params.mu)
        evals, evecs = np.linalg.eigh(H_k)
        Energies[i] = evals
        _, _, S_z[i] = calculate_spin_expectation(evecs)
    axes[0, 0].plot(k_x_vals, Energies)
    axes[0, 0].set_xlabel(r'$k_x$ (a$^{-1}$)')
    axes[0, 0].set_ylabel(r'$E$ (meV)')
    axes[0, 0].grid(True, linestyle=':')

    # Low energy plot
    for band in range(ham_dim):
        points = np.array([k_x_vals, Energies[:, band]]).T.reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)
        lc = LineCollection(segments, cmap='coolwarm', linewidth=2)
        segmentColors = 0.5 * (S_z[:-1, band] + S_z[1:, band])
        lc.set_array(segmentColors)
        axes[1, 0].add_collection(lc)
        axes[1, 0].autoscale()

    axes[1, 0].set_xlabel(r'$k_x$ (a$^{-1}$)')
    axes[1, 0].set_ylabel(r'$E$ (meV)')
    axes[1, 0].set_ylim(bottom=-1060, top = -900)
    axes[1, 0].grid(True, linestyle=':')

    k_y_vals = np.linspace(-MAX_KY_BZ, MAX_KY_BZ, n_k_points)
    Energies = np.zeros((k_y_vals.size, ham_dim))
    S_z = np.zeros((k_y_vals.size, ham_dim))
    for i, ky in enumerate(k_y_vals):
        H_k = compute_H(0, ky, params.t_sigma, params.t_pi, params.l_soc, params.t_rashba, params.d_tri, params.mu)
        evals, evecs = np.linalg.eigh(H_k)
        Energies[i] = evals
        _, _, S_z[i] = calculate_spin_expectation(evecs)
    axes[0, 1].plot(k_y_vals, Energies)
    axes[0, 1].set_xlabel(r'$k_y$ (a$^{-1}$)')
    axes[0, 1].set_ylabel(r'$E$ (meV)')
    axes[0, 1].grid(True, linestyle=':')

    # Low energy plot
    for band in range(ham_dim):
        points = np.array([k_y_vals, Energies[:, band]]).T.reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)
        lc = LineCollection(segments, cmap='coolwarm', linewidth=2)
        segmentColors = 0.5 * (S_z[:-1, band] + S_z[1:, band])
        lc.set_array(segmentColors)
        axes[1, 1].add_collection(lc)
        axes[1, 1].autoscale()

    axes[1, 1].set_xlabel(r'$k_y$ (a$^{-1}$)')
    axes[1, 1].set_ylabel(r'$E$ (meV)')
    axes[1, 1].set_ylim(bottom=-1060, top = -900)
    axes[1, 1].grid(True, linestyle=':')

    plt.tight_layout()
    plt.show()

plot_k_slices(params, 12)


In [ ]:
from scipy.interpolate import RegularGridInterpolator

def plot_dispersion(params: dispersion_parameters, ham_dim: int, n_k_points: int = 200, Fermi_levels: tuple = (-1000, -950)):
    k_x_vals = np.linspace(-MAX_KX_BZ, MAX_KX_BZ, n_k_points)
    k_y_vals = np.linspace(-MAX_KX_BZ, MAX_KX_BZ, n_k_points)
    KX, KY = np.meshgrid(k_x_vals, k_y_vals)
    Energies = np.zeros((n_k_points, n_k_points, ham_dim))
    S_x = np.zeros((n_k_points, n_k_points, ham_dim))
    S_y = np.zeros((n_k_points, n_k_points, ham_dim))
    S_z = np.zeros((n_k_points, n_k_points, ham_dim))
    for i, kx in enumerate(k_x_vals):
        for j, ky in enumerate(k_y_vals):
            H_k = compute_H(kx, ky, params.t_sigma, params.t_pi, params.l_soc, params.t_rashba, params.d_tri, params.mu)
            evals, evecs = np.linalg.eigh(H_k)
            Energies[i, j] = evals
            S_x[i, j], S_y[i, j], S_z[i, j] = calculate_spin_expectation(evecs)
    fig, axes = plt.subplots(2, 2, figsize=(20, 20))

    # First row: contourf plots
    axes[0, 0].contourf(KX, KY, Energies[:, :, 0])
    axes[0, 0].set_ylabel(r'$k_y$ (a$^{-1}$)')
    plotFirstBrillouinZoneBoundary(axes[0, 0])
    axes[0, 1].contourf(KX, KY, Energies[:, :, 1])
    plotFirstBrillouinZoneBoundary(axes[0, 1])

    # Second row: contour plots for each Fermi level
    for idx, level in enumerate(Fermi_levels):
        ax = axes[1, idx]
        for band in range(ham_dim):
            cs = ax.contour(KX, KY, Energies[:, :, band], levels=[level], linewidths=2)
            for vertices in cs.allsegs[0]:
                interp_S_x = RegularGridInterpolator((k_x_vals, k_y_vals),
                                                     S_x[:, :, band],
                                                     bounds_error=False,
                                                     fill_value=np.nan)
                interp_S_y = RegularGridInterpolator((k_x_vals, k_y_vals),
                                                     S_y[:, :, band],
                                                     bounds_error=False,
                                                     fill_value=np.nan)
                interp_S_z = RegularGridInterpolator((k_x_vals, k_y_vals),
                                                     S_z[:, :, band],
                                                     bounds_error=False,
                                                     fill_value=np.nan)

                interpolated_S_x = interp_S_x(vertices)
                interpolated_S_y = interp_S_y(vertices)
                interpolated_S_z = interp_S_z(vertices)

                kx_center, ky_center = vertices[:,0].mean(), vertices[:,1].mean()
                phases = np.arctan2(vertices[:,1] - ky_center, vertices[:,0] - kx_center)
                order = np.argsort(phases)

                vertices = vertices[order]
                interpolated_S_z = interpolated_S_z[order]

                segments = np.stack([vertices[:-1], vertices[1:]], axis=1)
                lc = LineCollection(segments, cmap='coolwarm', linewidth=2)
                lc.set_array(interpolated_S_z[:-1])
                lc.set_linewidth(2)
                ax.add_collection(lc)

                kx_c = vertices[:,0]
                ky_c = vertices[:,1]
                step = max(len(vertices) // 50, 1)
                kx_q = kx_c[::step]
                ky_q = ky_c[::step]
                Sx_q = interpolated_S_x[::step]
                Sy_q = interpolated_S_y[::step]
                Sz_q = interpolated_S_z[::step]
                ax.quiver(
                    kx_q, ky_q,
                    Sx_q, Sy_q,
                    Sz_q,                 # used for color
                    cmap="coolwarm",
                    scale=4,
                    scale_units="xy",
                    width=0.004,
                    headwidth=5,
                    headlength=4,
                    headaxislength=3,
                    zorder=3
                )
            ax.grid(True, linestyle=':')

        plotFirstBrillouinZoneBoundary(ax)

    axes[1, 0].set_ylabel(r'$k_y$ (a$^{-1}$)')
    plt.tight_layout()
    plt.show()

plot_dispersion(params, 12)

# Group theoretical approach to superconducting order parameter

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.append('/home/czarnecki/LAO-STO/GroupTheory')
from GroupGeneratorsClass import C3vGenerator
from SymbolicSymmetryProjectorClass import SymbolicSymmetryProjectorClass
from sympy.physics.quantum import TensorProduct
from sympy import latex
from IPython.display import Math, display

In [ ]:
#Generators

# Nearest neighbors subspace
c3_neigh = sp.Matrix([[0, 1, 0],
                      [0, 0, 1],
                      [1, 0, 0]])
s_v1_neigh = sp.Matrix([[0, 0, 1],
                        [0, 1, 0],
                        [1, 0, 0]])

# Orbital subspace
c3_orb = sp.Matrix([[0,1,0],
                    [0,0,1],
                    [1,0,0]])
s_v1_orb = sp.Matrix([[0,1,0],
                      [1,0,0],
                      [0,0,1]])

# Spin subspace
c3_spin = sp.Matrix([[1, 0, 0, 0],
                     [0, sp.cos(sp.pi * 2 / 3), -sp.sin(sp.pi * 2 / 3), 0],
                     [0, sp.sin(sp.pi * 2 / 3), sp.cos(sp.pi * 2 / 3), 0],
                     [0, 0, 0, 1]])
s_v1_spin = sp.Matrix([[1, 0, 0, 0],
                       [0, sp.Rational(1,2), sp.sqrt(3)/2, 0],
                       [0, sp.sqrt(3)/2, -sp.Rational(1,2), 0],
                       [0, 0, 0, 1]])


# Sublattice degree of freedom is excluded on purpose.
# This is because in the C_3v group no operation mixes sublattices, so that all operators
# in the sublattice space are unitary. This only extends the basis and generates (spare)
# solutions to the eigenproblem.
# We take care of the number of sublattices later on when constructing the pairing matrix.

c3_total = TensorProduct(c3_spin, c3_orb, c3_neigh)
s_v1_total = TensorProduct(s_v1_spin, s_v1_orb, s_v1_neigh)

c3vGroup = C3vGenerator(c3_total, s_v1_total)

symmetryResolver = SymbolicSymmetryProjectorClass(c3vGroup.irrepsTuple,
                                                         c3vGroup.conjugacyClassesTuple,
                                                         c3vGroup.chiTabDict,
                                                         c3vGroup.representationDim)
projectionMatrices = symmetryResolver.getProjectionOperators(c3vGroup.getOperationsDict())
multiplicities = symmetryResolver.getMultiplicities(c3vGroup.getOperationsDict())
for irrep in multiplicities:
  print(irrep)
  display(multiplicities[irrep])
eigenproblem = symmetryResolver.getDiagonalizedProjections(projectionMatrices)
eigenproblem = symmetryResolver.filterOutProjections(eigenproblem)
# for irrep in c3vGroup.irrepsTuple:
#   print(irrep)
#   for i,  solution in enumerate(eigenproblem[irrep]):
#       display(f"Eigenvalue: {solution[0]}")
#       display(f"Multiplicity: {solution[1]}")
#       latex_line = ",\; ".join(latex(evec) for evec in solution[2])
#       display(Math(latex_line))
# symmetryResolver.displayProjectionsMetadata(eigenproblem)


In [ ]:
def get_matrix_indices(projectionIndex: int) -> tuple[int, int]:
  """
  Returns indices of matrix elements in 12x12 matrix for given projection index.
  """
  spin = projectionIndex // 3
  orbital = projectionIndex % 3

  return spin, orbital

proj_eigvec = eigenproblem['A_1'][0][2][0]
proj_eigvec = proj_eigvec.reshape(12, 3)

orb_yz = sp.Matrix([[1, 0, 0],
                    [0, 0, 0],
                    [0, 0, 0]])
orb_zx = sp.Matrix([[0, 0, 0],
                    [0, 1, 0],
                    [0, 0, 0]])
orb_xy = sp.Matrix([[0, 0, 0],
                    [0, 0, 0],
                    [0, 0, 1]])

sublat_hop_12 = sp.Matrix([[0, 1],
                           [0, 0]])


basis_functions_12 = sp.Matrix([sp.exp(sp.I * k1), sp.exp(sp.I * k2), sp.exp(sp.I * k3)])

orb_couplings = [orb_yz, orb_zx, orb_xy]
spin_parts = [s_0, s_x, s_y, s_z]


pairing_matrices_ir_dict: dict[str, list[sp.Matrix]] = {}
for irrep in c3vGroup.irrepsTuple:
  # Second index is zero, as we are left only with single eigenvalue == 1 after filtering out the others.
  ir_duplicate_count = 0
  deltas_ir = []
  for proj_eigvec in eigenproblem[irrep][0][2]:
    proj_eigvec = proj_eigvec.reshape(12, 3) # each row represent three nearest neighbors in a given sublattice.
    delta_k = sp.zeros(12, 12)

    sublat_idx_1_detected = False
    for i in range(len(proj_eigvec[:, 0])):
      spin, orbital = get_matrix_indices(i)
      #display(f"Indices {spin}, {sublattice}, {orbital}")
      #display(proj_eigvec[i, :])

      elem_k_space = proj_eigvec[i, :].dot(basis_functions_12)
      elem_matrix = TensorProduct(-sp.I * s_y * spin_parts[spin], sublat_hop_12, orb_couplings[orbital])
      elem_matrix *= elem_k_space

      delta_k += elem_matrix

    # Antisymmetrize the gap matrix
    delta_minus_k = delta_k
    delta_minus_k = delta_minus_k.subs(k_x, -k_x)
    delta_minus_k = delta_minus_k.subs(k_y, -k_y)

    delta_anitsymmetrized = (delta_k - delta_minus_k.T) # TODO: Verify that this should be multiplied by 1/2

    # display(delta_k)
    # display(delta_minus_k)
    # display(delta_anitsymmetrized)
    deltas_ir.append(delta_anitsymmetrized)
    # Check fermionic antisymmetry
  print(f"Number of deltas for irrep {irrep}: {len(deltas_ir)}")
  pairing_matrices_ir_dict[irrep] = deltas_ir


In [ ]:
for irrep in c3vGroup.irrepsTuple:
  for matrix in pairing_matrices_ir_dict[irrep]:
    display(matrix)


In [ ]:
nambu_hamiltonian_callbacks: dict = {}

for irrep in c3vGroup.irrepsTuple:
  pairing_matrices = pairing_matrices_ir_dict[irrep]
  coeffs_symbols = [sp.Symbol(f"xi_{i}", complex=True) for i in range(len(pairing_matrices))]
  gamma_k: sp.Matrix = sp.zeros(12, 12)
  for i in range(len(pairing_matrices)):
    gamma_k += coeffs_symbols[i] * pairing_matrices[i]

  H_k = H
  H_minus_k = H_k.subs(k_x, -k_x)
  H_minus_k = H_minus_k.subs(k_y, -k_y)
  H_minus_k = H_minus_k.T
  H_minus_k = -H_minus_k
  H_nambu = sp.Rational(1,2) * sp.BlockMatrix([[H_k, gamma_k], [sp.conjugate(gamma_k.T), H_minus_k]])

  # Assigning a callback that will build the Nambu Hamiltonian for each irrep
  nambu_hamiltonian_callbacks[irrep] = sp.lambdify((k_x, k_y, t_sigma, t_pi, l_soc, t_rashba, d_tri, mu, *coeffs_symbols), H_nambu, modules='numpy')


In [ ]:
#Integration constants
R_K_MAX = 4 * np.pi / (3 * np.sqrt(3.0))
N_POINTS = 30
N_REFINEMENTS = 10

In [ ]:
from tqdm import tqdm
import numba as nb

def calculate_dos(params, irrep, ir_coeffs, E_min, E_max, dE, zeta, n_k_points, n_refinements):
  @nb.njit(fastmath=True)
  def accumulate_dos(DOS, Energies, eigvals, zeta):
    inv_pi = 1.0 / np.pi
    z2 = zeta * zeta

    for n in range(eigvals.size):
      E = eigvals[n]
      for i in range(Energies.size):
          d = Energies[i] - E
          DOS[i, n] += zeta * inv_pi / (d*d + z2)



  Hamiltonian = nambu_hamiltonian_callbacks[irrep](0,
                                                   0,
                                                   params.t_sigma,
                                                   params.t_pi,
                                                   params.l_soc,
                                                   params.t_rashba,
                                                   params.d_tri,
                                                   params.mu,
                                                   *ir_coeffs
                                                 )
  ham_dim = len(Hamiltonian)
  Energies = np.arange(E_min, E_max, dE)
  DOS = np.zeros((len(Energies), ham_dim))

  for n_triangle in range(6):
    print(f"Triangle {n_triangle + 1} / {6}")
    phi_min = n_triangle * np.pi / 3
    phi_max = (n_triangle + 1) * np.pi / 3
    d_phi = (phi_max - phi_min) / n_k_points
    for i_phi in tqdm(range(n_k_points)):
      phi_k = phi_min + i_phi * d_phi
      phi_k_relative_to_triangle_boundary = np.mod(phi_k, np.pi / 3)
      r_max = R_K_MAX * np.sqrt(3.0) / (2 * np.cos(phi_k_relative_to_triangle_boundary - np.pi/6))

      for j_r in range(n_k_points):
        d_r = r_max / n_k_points
        r_k = j_r * d_r

        k_x = r_k * np.cos(phi_k)
        k_y = r_k * np.sin(phi_k)

        Hamiltonian = nambu_hamiltonian_callbacks[irrep](k_x,
                                                         k_y,
                                                         params.t_sigma,
                                                         params.t_pi,
                                                         params.l_soc,
                                                         params.t_rashba,
                                                         params.d_tri,
                                                         params.mu,
                                                         *ir_coeffs
                                                        )
        eigvals = np.linalg.eigvalsh(Hamiltonian)
        accumulate_dos(DOS, Energies, eigvals, zeta)


        # Grid refinement
        # Condition that we have no in-gap states
        if np.min(eigvals) > E_min:
          continue

        d_phi_refined = d_phi / n_refinements
        for i_phi_refined in range(n_refinements):
          phi_k_refined = phi_k + i_phi_refined * d_phi_refined
          for j_r_refined in range(n_refinements):
            phi_k_relative_to_triangle_boundary = np.mod(phi_k_refined, np.pi / 3)
            r_max_refined = R_K_MAX * np.sqrt(3.0) / (2 * np.cos(phi_k_relative_to_triangle_boundary - np.pi/6))
            d_r = r_max_refined / n_refinements
            r_k_refined = j_r_refined * d_r

            if r_k_refined > r_max or phi_k_refined > phi_max:
              continue

            k_x = r_k_refined * np.cos(phi_k_refined)
            k_y = r_k_refined * np.sin(phi_k_refined)

            Hamiltonian = nambu_hamiltonian_callbacks[irrep](k_x,
                                                            k_y,
                                                            params.t_sigma,
                                                            params.t_pi,
                                                            params.l_soc,
                                                            params.t_rashba,
                                                            params.d_tri,
                                                            params.mu,
                                                            *ir_coeffs
                                                            )
            eigvals = np.linalg.eigvalsh(Hamiltonian)
            accumulate_dos(DOS, Energies, eigvals, zeta)

  return Energies, DOS

def plot_dos(Energies, DOS, ax=None):
  if ax == None:
    fig, ax = plt.subplots()
  dos_normalization = np.max(np.sum(DOS, axis=1))
  ax.plot(Energies, DOS / dos_normalization)
  ax.plot(Energies, np.sum(DOS, axis=1) / dos_normalization, color='black', linewidth=3)
  ax.set_xlabel("E (meV)")
  ax.set_ylabel("DOS")
  return ax


In [ ]:
from dataclasses import astuple
#Test
params = dispersion_parameters() # default values
params.mu = -1030
irrep = "A_1"
coeffs = np.array([np.complex128(0.) for _ in range(len(pairing_matrices_ir_dict[irrep]))])
coeffs[0] = 0.1
coeffs[1] = 0.5
Energies, DOS = calculate_dos(params, irrep, coeffs, -1, 1, 1e-2, 1e-2, N_POINTS, N_REFINEMENTS)
plot_dos(Energies, DOS)

In [ ]:
from scipy.integrate import quad, fixed_quad


# Function that will calculate free energy - this is the function we actually want to minimize!
def calculate_free_energy(ir_coeffs: np.ndarray, args: tuple) -> np.float64:

  # Integrate free energy over 1 BZ in polar coordinates
  kx_points = []
  ky_points = []

  def integrate_over_r(r_k, *args):
    phi_k, t_sigma, t_pi, l_soc, t_rashba, d_tri, mu, ir_coeffs = args
  
    k_x = r_k * np.cos(phi_k)
    k_y = r_k * np.sin(phi_k)

    kx_points.append(k_x)
    ky_points.append(k_y)

    # Hamiltonian = nambu_hamiltonian_callbacks[irrep](k_x, k_y, t_sigma, t_pi, l_soc, t_rashba, d_tri, mu, *ir_coeffs)
    # dim_half = Hamiltonian.shape[0] // 2
    # gamma_matrix = Hamiltonian[:dim_half, dim_half:]
    # condensation_energy = np.trace(np.conjugate(gamma_matrix.T) @ gamma_matrix)
    # Energies = np.linalg.eigvalsh(Hamiltonian)
    # Energies_plus = Energies[dim_half:]
    # free_energy = condensation_energy - 0.5 * np.sum(Energies_plus) #In T -> 0 limit
    # TODO: add Jacobian multiplication
    return r_k

  def integrate_over_phi(phi_k, *args):
    t_sigma, t_pi, l_soc, t_rashba, d_tri, mu, ir_coeffs = args
    phi_k_relative_to_triangle_boundary = np.mod(phi_k, np.pi / 3)
    r_max = R_K_MAX * np.sqrt(3.0) / (2 * np.cos(phi_k_relative_to_triangle_boundary - np.pi/6))
    r_vals = np.linspace(0, r_max, N_POINTS)
    d_r = r_vals[1] - r_vals[0]
    free_energy_phi = 0
    for r_k in r_vals:
      free_energy_phi += integrate_over_r(r_k, phi_k, t_sigma, t_pi, l_soc, t_rashba, d_tri, mu, ir_coeffs)

    return free_energy_phi * d_r


  irrep, t_sigma, t_pi, l_soc, t_rashba, d_tri, mu = args #Unpack arguments
  free_energy = 0

  for n_triangle in range(6):
    phi_min = n_triangle * np.pi / 3
    phi_max = (n_triangle + 1) * np.pi / 3

    phi_vals = np.linspace(phi_min, phi_max, N_POINTS)
    d_phi = phi_vals[1] - phi_vals[0]
    for phi_k in phi_vals:
      free_energy += d_phi * integrate_over_phi(phi_k, t_sigma, t_pi, l_soc, t_rashba, d_tri, mu, ir_coeffs)

  fig, ax = plt.subplots()
  ax.scatter(kx_points, ky_points)
  plotFirstBrillouinZoneBoundary(ax)
  plt.show()

  return free_energy

In [ ]:
# #Test
# params = dispersion_parameters() # default values
# params.mu = -1030
# irrep = "A_1"
# coeffs = np.array([np.complex128(1) for _ in range(len(pairing_matrices_ir_dict[irrep]))])
# free_energy = calculate_free_energy(coeffs, (irrep, params.t_sigma, params.t_pi, params.l_soc, params.t_rashba, params.d_tri, params.mu))
# print(f"Free energy: {free_energy}")
# print(f"Jacobian: {8 * np.pi**2 / (3 * np.sqrt(3))}")

# first_eigvec_pairing_coeffs = np.linspace(0, 0.01, 10)
# free_energies = []
# for coeff in first_eigvec_pairing_coeffs:
#   coeffs[0] = coeff
#   free_energies.append(calculate_free_energy(coeffs, (irrep, params.t_sigma, params.t_pi, params.l_soc, params.t_rashba, params.d_tri, params.mu)))
#   print(f"Free energy for coeff {coeff}: {free_energies[-1]}")
# plt.plot(first_eigvec_pairing_coeffs, free_energies)
# plt.xlabel("Pairing coefficient")
# plt.ylabel("Free energy")
# plt.show()
